# Assignment 6.1 - Decision Trees and Random Forests on Car Data

This notebook contains two activities from Chapter 4:

* **Activity 4.01** - Build a decision tree classifier that rates cars as unacceptable, acceptable, good or very good.
* **Activity 4.02** - Optimize the classifier with a Random Forest and an Extra Trees model, and compare them.

Both use the Car Evaluation dataset. It has six features (buying, maintenance, doors, persons, luggage_boot, safety) and one target column, class, with four values: unacc, acc, good, vgood.

# Activity 4.01: Car Data Classification

## Step 1 - Import the libraries and load the dataset
We import pandas and load the car dataset from the textbook's GitHub repository into a DataFrame. Printing the first five rows confirms the data loaded correctly.

In [1]:
# pandas loads and handles the dataset as a table (DataFrame)
import pandas as pd

# URL of the raw car dataset in the textbook's GitHub repository
file_url = 'https://raw.githubusercontent.com/'\
           'PacktWorkshops/'\
           'The-Applied-Artificial-Intelligence-Workshop/'\
           'master/Datasets/car.csv'

# Load the CSV file into a DataFrame
df = pd.read_csv(file_url)

# Show the first five rows and the size (rows, columns) of the data
print(df.head())
print("\nDataset shape (rows, columns):", df.shape)

  buying maintenance doors persons luggage_boot safety  class
0  vhigh       vhigh     2       2        small    low  unacc
1  vhigh       vhigh     2       2        small    med  unacc
2  vhigh       vhigh     2       2        small   high  unacc
3  vhigh       vhigh     2       2          med    low  unacc
4  vhigh       vhigh     2       2          med    med  unacc

Dataset shape (rows, columns): (1728, 7)


## Step 2 - Label encoding
Every column in this dataset holds text (for example vhigh, small, unacc), but scikit-learn models only work with numbers. LabelEncoder converts each text category into an integer. We write one small function, encode(), and apply it to every column with a loop.

In [2]:
# preprocessing contains LabelEncoder (text categories -> numbers)
from sklearn import preprocessing

# Helper function: encode one column and return the encoded values
def encode(data_frame, column):
    # create the encoder
    label_encoder = preprocessing.LabelEncoder()
    # learn the unique categories found in the column
    label_encoder.fit(data_frame[column].unique())
    # replace each category with its number
    return label_encoder.transform(data_frame[column])

# Apply the encoder to every column (features and target)
for column in df.columns:
    df[column] = encode(df, column)

# Show the first five rows again - every value is now a number
print(df.head())

   buying  maintenance  doors  persons  luggage_boot  safety  class
0       3            3      0        0             2       1      2
1       3            3      0        0             2       2      2
2       3            3      0        0             2       0      2
3       3            3      0        0             1       1      2
4       3            3      0        0             1       2      2


## Step 3 - Extract the label (target) variable
The column we want to predict is class. Using pop() removes it from the DataFrame and stores it in a separate variable called label. What remains in df are only the six features.

In [3]:
# pop() removes the 'class' column from df and returns it:
# label = target, df = features only
label = df.pop('class')

print("Label (first 5 values):")
print(label.head())
print("\nRemaining feature columns:")
print(", ".join(df.columns))

Label (first 5 values):
0    2
1    2
2    2
3    2
4    2
Name: class, dtype: int64

Remaining feature columns:
buying, maintenance, doors, persons, luggage_boot, safety


## Step 4 - Split into training and testing data
We keep 90% of the data to train the model and hold back 10% as unseen test data to measure how well the model generalizes. random_state=88 makes the split identical every time the notebook is run.

In [4]:
# model_selection contains train_test_split
from sklearn import model_selection

# Split features (df) and target (label): 90% training, 10% testing
features_train, features_test, label_train, label_test = \
    model_selection.train_test_split(df, label,
                                     test_size=0.1,
                                     random_state=88)

print("Training set size:", features_train.shape)
print("Testing set size: ", features_test.shape)

Training set size: (1555, 6)
Testing set size:  (173, 6)


## Step 5 - Build the decision tree classifier
A decision tree learns a series of yes/no questions about the features to decide which class a car belongs to. We create the classifier and train it with fit() on the training data.

Note: random_state=2 is set so the tree is built the same way on every run. Without it, scikit-learn breaks ties between equally good splits randomly, and the accuracy changes slightly each time the notebook is executed.

In [5]:
# DecisionTreeClassifier builds the decision tree model
from sklearn.tree import DecisionTreeClassifier

# Create the classifier (random_state makes the result reproducible)
decision_tree = DecisionTreeClassifier(random_state=2)

# Train (fit) the tree on the training data
decision_tree.fit(features_train, label_train)

# Show the trained model and its settings
print(decision_tree)

DecisionTreeClassifier(random_state=2)


## Step 6 - Score the model on the test data
score() predicts the cars in the test set and returns the accuracy: the fraction of cars classified correctly.

In [6]:
# Accuracy of the decision tree on the unseen test data
accuracy = decision_tree.score(features_test, label_test)
print("Decision tree accuracy on the test set:", accuracy)

Decision tree accuracy on the test set: 0.9595375722543352


## Step 7 - Deeper evaluation with a classification report
Accuracy alone can hide problems with individual classes. classification_report() shows, for each class, the precision (of the cars predicted as this class, how many really were), the recall (of the cars that really are this class, how many were found) and the F1-score (the balance of the two).

Class numbers after encoding: 0 = acc, 1 = good, 2 = unacc, 3 = vgood.

In [7]:
# classification_report gives precision, recall and F1 per class
from sklearn.metrics import classification_report

# Predict the classes of the test cars with the decision tree
dt_predictions = decision_tree.predict(features_test)

# Print the report comparing the true labels with the predictions
print(classification_report(label_test, dt_predictions))

              precision    recall  f1-score   support

           0       0.89      0.98      0.93        42
           1       0.89      0.89      0.89         9
           2       0.99      0.97      0.98       114
           3       1.00      0.75      0.86         8

    accuracy                           0.96       173
   macro avg       0.94      0.90      0.92       173
weighted avg       0.96      0.96      0.96       173



### Conclusion - Activity 4.01
The decision tree reaches an accuracy of about **0.96** on the test data, which is strong for a first attempt. Precision is high for all four classes (0.89 to 1.00). Recall is weakest for class 3 (vgood) at 0.75, meaning some very good cars are mistaken for another class. This is partly because class 3 is rare in the data (only 8 test cars), so every miss weighs heavily.

# Activity 4.02: Random Forest Classification for Your Car Rental Company

**Step 1 - Reuse Steps 1-4 of Activity 4.01.** The data has already been loaded, encoded, separated into features and label, and split into training and testing sets above, so the same features_train, features_test, label_train and label_test are reused here.

A single decision tree tends to overfit. A **random forest** trains many trees on random samples of the data and lets them vote, which usually generalizes better. An **extra trees** model goes one step further by also choosing the split points at random.

## Random Forest

### Steps 2-3 - Create and train the random forest
We create a forest of 100 trees, each limited to a depth of 6, and train it with fit(). random_state=168 makes the result reproducible.

In [8]:
# RandomForestClassifier builds many trees and combines their votes
from sklearn.ensemble import RandomForestClassifier

# 100 trees, each at most 6 levels deep
random_forest_classifier = RandomForestClassifier(n_estimators=100,
                                                  max_depth=6,
                                                  random_state=168)

# Train the forest on the training data
random_forest_classifier.fit(features_train, label_train)

print(random_forest_classifier)

RandomForestClassifier(max_depth=6, random_state=168)


### Predictions on the test set
We use the trained forest to predict the class of every car in the test set.

In [9]:
# Predict the classes of the test cars with the random forest
rf_preds_test = random_forest_classifier.predict(features_test)

print("Random forest predictions on the test set:")
print(rf_preds_test)

Random forest predictions on the test set:
[0 0 2 0 0 2 3 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 0 2 2 3 2 2 2 2 0 2 0 2 2 2 2
 2 2 2 2 2 2 2 2 0 2 0 0 0 2 2 0 2 0 2 0 2 2 2 0 2 2 2 2 0 0 2 2 2 0 2 3 2
 2 2 0 2 2 2 2 2 0 0 0 2 2 2 2 2 2 2 0 2 0 0 2 2 0 2 2 2 0 0 2 0 2 2 2 2 0
 2 2 0 2 2 3 0 2 2 2 2 0 0 2 2 2 0 2 2 2 0 2 2 2 2 2 2 2 0 3 3 2 0 0 2 2 2
 0 0 2 2 0 2 2 2 2 2 0 0 2 2 2 0 2 2 2 2 0 2 0 0 2]


### Steps 4-5 - Confusion matrix and classification report
The **confusion matrix** counts, for each true class (rows), how many cars were predicted as each class (columns). Numbers on the diagonal are correct predictions; everything else is a mistake. The classification report then summarizes precision, recall and F1-score per class.

In [10]:
# confusion_matrix shows which classes the model confuses
from sklearn.metrics import confusion_matrix

# zero_division=0 only hides a warning for a class that is never
# predicted; it does not change any score
print("Classification report - Random Forest")
print(classification_report(label_test, rf_preds_test,
                            zero_division=0))

print("Confusion matrix - Random Forest")
print("(rows = true class, columns = predicted class)")
print(confusion_matrix(label_test, rf_preds_test))

Classification report - Random Forest
              precision    recall  f1-score   support

           0       0.67      0.76      0.71        42
           1       0.00      0.00      0.00         9
           2       0.92      0.96      0.94       114
           3       0.83      0.62      0.71         8

    accuracy                           0.84       173
   macro avg       0.60      0.59      0.59       173
weighted avg       0.80      0.84      0.82       173

Confusion matrix - Random Forest
(rows = true class, columns = predicted class)
[[ 32   0  10   0]
 [  8   0   0   1]
 [  5   0 109   0]
 [  3   0   0   5]]


### Step 6 - Feature importance
feature_importances_ tells us how much each feature contributed to the forest's decisions. The values add up to 1, and a higher value means a more influential feature.

In [11]:
# Importance score of each feature (same order as the data columns)
rf_varimp = random_forest_classifier.feature_importances_
print("Random forest feature importance (raw array):")
print(rf_varimp)

# Pair each score with its feature name, most important first
print("\nFeature importance by name:")
rf_series = pd.Series(rf_varimp, index=features_train.columns)
print(rf_series.sort_values(ascending=False))

Random forest feature importance (raw array):
[0.12676384 0.10366314 0.02119621 0.35266673 0.05915769 0.33655239]

Feature importance by name:
persons         0.352667
safety          0.336552
buying          0.126764
maintenance     0.103663
luggage_boot    0.059158
doors           0.021196
dtype: float64


## Extra Trees (Extremely Randomized Trees)

### Step 7 - Repeat Steps 2 to 6 with an Extra Trees model
We repeat the same steps with ExtraTreesClassifier, using the same settings (100 trees, depth 6, random_state=168) so the two models can be compared fairly.

In [12]:
# ExtraTreesClassifier is like a random forest, but it also picks
# the split points at random
from sklearn.ensemble import ExtraTreesClassifier

# Same settings as the random forest for a fair comparison
extra_trees_classifier = ExtraTreesClassifier(n_estimators=100,
                                              max_depth=6,
                                              random_state=168)

# Train the extra trees model on the training data
extra_trees_classifier.fit(features_train, label_train)

print(extra_trees_classifier)

ExtraTreesClassifier(max_depth=6, random_state=168)


In [13]:
# Predict the classes of the test cars with the extra trees model
et_preds_test = extra_trees_classifier.predict(features_test)

print("Extra trees predictions on the test set:")
print(et_preds_test)

Extra trees predictions on the test set:
[0 0 2 0 2 2 3 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 0 2 2 0 2 2 2 2 0 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 0 2 0 0 0 2 2 2 2 0 2 0 2 2 2 0 2 2 2 2 0 0 2 2 2 2 2 0 2
 2 2 0 2 2 2 2 2 0 0 0 2 2 2 2 2 2 2 0 2 0 0 2 2 0 2 2 2 2 0 2 0 2 2 2 2 0
 2 2 0 2 2 0 0 2 2 2 2 0 2 2 2 2 0 2 2 2 0 2 2 2 2 2 2 2 0 0 0 2 0 0 2 2 2
 0 0 2 2 0 2 2 2 2 2 0 0 2 2 2 0 2 2 2 2 2 2 0 0 2]


In [14]:
# Classification report and confusion matrix for extra trees
print("Classification report - Extra Trees")
print(classification_report(label_test, et_preds_test,
                            zero_division=0))

print("Confusion matrix - Extra Trees")
print("(rows = true class, columns = predicted class)")
print(confusion_matrix(label_test, et_preds_test))

Classification report - Extra Trees
              precision    recall  f1-score   support

           0       0.61      0.67      0.64        42
           1       0.00      0.00      0.00         9
           2       0.89      0.98      0.93       114
           3       1.00      0.12      0.22         8

    accuracy                           0.82       173
   macro avg       0.62      0.44      0.45       173
weighted avg       0.78      0.82      0.78       173

Confusion matrix - Extra Trees
(rows = true class, columns = predicted class)
[[ 28   0  14   0]
 [  9   0   0   0]
 [  2   0 112   0]
 [  7   0   0   1]]


In [15]:
# Feature importance of the extra trees model
et_varimp = extra_trees_classifier.feature_importances_
print("Extra trees feature importance (raw array):")
print(et_varimp)

# Pair each score with its feature name, most important first
print("\nFeature importance by name:")
et_series = pd.Series(et_varimp, index=features_train.columns)
print(et_series.sort_values(ascending=False))

Extra trees feature importance (raw array):
[0.08844544 0.0702334  0.01440408 0.37662014 0.05965896 0.39063797]

Feature importance by name:
safety          0.390638
persons         0.376620
buying          0.088445
maintenance     0.070233
luggage_boot    0.059659
doors           0.014404
dtype: float64


### Conclusion - Activity 4.02
* **Random forest:** accuracy 0.84 and a weighted F1-score of 0.82. It is strong on class 2 (unacc, the most common class) but struggles with class 1 (good), which it never predicts correctly. The confusion matrix shows its main weakness is class 0 (acc): 10 real acc cars were predicted as unacc, and 16 cars from other classes were wrongly predicted as acc.
* **Extra trees:** accuracy 0.82 and a weighted F1-score of 0.78, so it performed slightly worse than the random forest here. It finds almost none of the rare classes (1 and 3).
* **Feature importance:** both models agree that the two most influential features are safety and persons (roughly 0.34-0.39 each), while doors matters least (about 0.01-0.02). Safety and passenger capacity are what drive a car's rating.
* **Comparison with Activity 4.01:** the single decision tree (accuracy about 0.96) scored higher than both ensembles on this test set. A likely reason is that the ensembles here are limited to a depth of 6, which makes it harder for them to fit the rare classes. They could probably be improved by tuning hyperparameters such as max_depth and n_estimators.